# 🎬 Chrome Extension → Colab 무료 영상 편집 파이프라인\n\n**Chrome Extension에서 수집한 이미지를 Colab에서 무료로 영상 편집**\n\n## 📌 주요 기능:\n- Chrome Extension 이미지 → Google Drive 업로드\n- MoviePy로 무료 영상 편집 (트림, 효과, 전환, 자막)\n- AI 기반 향상 (업스케일, 프레임 보간, 스타일 변환)\n- 최종 영상 다운로드 및 공유\n\n---\n\n### ⚙️ 실행 전 필수\n1. **런타임 → 런타임 유형 변경 → GPU (T4)** 설정\n2. Chrome Extension에서 이미지 수집 완료\n3. Google 계정 로그인 (Drive 연동용)

## 1️⃣ 환경 설정 및 Drive 마운트

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 설정
import os
WORK_DIR = '/content/drive/MyDrive/ChromeExtension_Videos'
os.makedirs(f'{WORK_DIR}/uploads', exist_ok=True)
os.makedirs(f'{WORK_DIR}/processed', exist_ok=True)
os.makedirs(f'{WORK_DIR}/output', exist_ok=True)
print(f"✅ 작업 디렉토리: {WORK_DIR}")

## 2️⃣ Chrome Extension 연동 설정

In [ ]:
# Extension 연동 ID 생성 (고유 세션 ID)
import uuid
import json
from datetime import datetime

SESSION_ID = str(uuid.uuid4())[:8]
config = {
    "session_id": SESSION_ID,
    "created_at": datetime.now().isoformat(),
    "drive_folder": f"{WORK_DIR}/uploads/{SESSION_ID}",
    "colab_notebook": "colab_video_editor.ipynb",
    "status": "ready"
}

# 설정 파일 저장
os.makedirs(config['drive_folder'], exist_ok=True)
with open(f"{WORK_DIR}/session_{SESSION_ID}.json", 'w') as f:
    json.dump(config, f, indent=2)

print("🔗 Chrome Extension 연동 정보:")
print(f"Session ID: {SESSION_ID}")
print(f"Upload folder: {config['drive_folder']}")
print("\n📋 Extension 설정에 위 Session ID를 입력하세요")

## 3️⃣ 필수 라이브러리 설치

In [ ]:
# MoviePy 및 편집 도구 설치
!pip install -q moviepy pillow numpy opencv-python-headless
!pip install -q imageio imageio-ffmpeg gtts pydub
!apt-get update && apt-get install -y ffmpeg fonts-noto-cjk

import moviepy.editor as mpe
from moviepy.editor import *
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont
import glob
from IPython.display import Video, display, HTML

print("✅ 라이브러리 설치 완료")

## 4️⃣ Extension 이미지 불러오기

In [ ]:
# Chrome Extension에서 업로드한 이미지 확인
import base64
from io import BytesIO

def load_extension_images(session_id):
    """Extension에서 업로드한 이미지 로드"""
    folder = f"{WORK_DIR}/uploads/{session_id}"
    
    # JSON 파일에서 이미지 데이터 로드
    json_file = f"{folder}/images.json"
    if os.path.exists(json_file):
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        images = []
        for idx, img_data in enumerate(data.get('images', [])):
            if img_data['url'].startswith('data:'):
                # Data URL 디코딩
                header, encoded = img_data['url'].split(',', 1)
                img_bytes = base64.b64decode(encoded)
                img = Image.open(BytesIO(img_bytes))
            else:
                # URL에서 다운로드
                import requests
                response = requests.get(img_data['url'])
                img = Image.open(BytesIO(response.content))
            
            # 이미지 저장
            img_path = f"{folder}/image_{idx:03d}.png"
            img.save(img_path)
            images.append(img_path)
            
        return images
    
    # 또는 직접 업로드된 이미지 파일들
    images = sorted(glob.glob(f"{folder}/*.{png,jpg,jpeg}"))
    return images

# 이미지 로드
image_files = load_extension_images(SESSION_ID)
print(f"📸 로드된 이미지: {len(image_files)}개")

# 썸네일 표시
if image_files:
    from IPython.display import Image as IPImage
    display(IPImage(image_files[0], width=300))
    print(f"첫 번째 이미지: {image_files[0]}")

## 5️⃣ 영상 편집 설정

In [ ]:
# ===== 편집 설정 (자유롭게 수정) =====
VIDEO_CONFIG = {
    # 기본 설정
    "duration_per_image": 3.0,      # 이미지당 표시 시간(초)
    "fps": 30,                       # 프레임 레이트
    "resolution": (1920, 1080),     # 해상도 (width, height)
    
    # 전환 효과
    "transition_type": "crossfade",  # crossfade, slide, zoom, rotate
    "transition_duration": 0.5,      # 전환 시간(초)
    
    # 모션 효과
    "enable_ken_burns": True,        # Ken Burns 효과 (줌/패닝)
    "zoom_ratio": 1.2,               # 줌 비율
    
    # 텍스트/자막
    "add_captions": True,            # 자막 추가
    "caption_position": "bottom",    # top, center, bottom
    "font_size": 48,
    "font_color": "white",
    "stroke_color": "black",
    "stroke_width": 2,
    
    # 필터/색보정
    "color_filter": "cinematic",     # none, cinematic, vintage, cold, warm
    "brightness": 1.0,               # 밝기 (0.5 ~ 2.0)
    "contrast": 1.1,                 # 대비 (0.5 ~ 2.0)
    "saturation": 1.2,               # 채도 (0 ~ 2.0)
    
    # 오디오
    "bgm_url": None,                 # 배경음악 URL 또는 경로
    "bgm_volume": 0.3,               # BGM 볼륨 (0 ~ 1)
    "fade_audio": True,              # 오디오 페이드 인/아웃
    
    # 출력
    "output_format": "mp4",          # mp4, avi, mov, webm
    "codec": "libx264",              # 비디오 코덱
    "quality": "high",               # low, medium, high, ultra
}

# 품질 프리셋
QUALITY_PRESETS = {
    "low": {"bitrate": "2M", "crf": 28},
    "medium": {"bitrate": "5M", "crf": 23},
    "high": {"bitrate": "10M", "crf": 18},
    "ultra": {"bitrate": "20M", "crf": 15}
}

print("⚙️ 영상 편집 설정 완료")
print(f"총 영상 길이: 약 {len(image_files) * VIDEO_CONFIG['duration_per_image']}초")

## 6️⃣ 영상 편집 함수들

In [ ]:
def create_ken_burns_effect(image_path, duration, zoom_ratio=1.2):
    """Ken Burns 효과 (줌/패닝) 생성"""
    clip = ImageClip(image_path, duration=duration)
    w, h = clip.size
    
    # 랜덤 시작/끝 위치
    start_zoom = 1.0
    end_zoom = zoom_ratio
    
    def zoom_effect(get_frame, t):
        frame = get_frame(t)
        zoom = start_zoom + (end_zoom - start_zoom) * (t / duration)
        new_w, new_h = int(w * zoom), int(h * zoom)
        
        # 중앙 크롭
        frame = cv2.resize(frame, (new_w, new_h))
        x = (new_w - w) // 2
        y = (new_h - h) // 2
        return frame[y:y+h, x:x+w]
    
    return clip.fl(zoom_effect)

def apply_color_filter(clip, filter_type="cinematic"):
    """색상 필터 적용"""
    filters = {
        "cinematic": lambda img: img.fx(vfx.colorx, 1.2).fx(vfx.lum_contrast, 0, 20, 128),
        "vintage": lambda img: img.fx(vfx.blackwhite).fx(vfx.colorx, 1.5),
        "cold": lambda img: img.fx(vfx.colorx, 0.8, 0.8, 1.2),
        "warm": lambda img: img.fx(vfx.colorx, 1.2, 1.1, 0.9),
        "none": lambda img: img
    }
    
    if filter_type in filters:
        return filters[filter_type](clip)
    return clip

def add_transition(clip1, clip2, transition_type="crossfade", duration=0.5):
    """클립 간 전환 효과"""
    if transition_type == "crossfade":
        return CompositeVideoClip([
            clip1,
            clip2.set_start(clip1.duration - duration).crossfadein(duration)
        ])
    elif transition_type == "slide":
        return concatenate_videoclips([
            clip1.fx(vfx.slide_out, duration, 'right'),
            clip2.fx(vfx.slide_in, duration, 'left')
        ])
    else:
        return concatenate_videoclips([clip1, clip2])

def add_text_overlay(clip, text, position="bottom", **kwargs):
    """텍스트 오버레이 추가"""
    txt_clip = TextClip(
        text,
        fontsize=kwargs.get('font_size', 48),
        color=kwargs.get('font_color', 'white'),
        stroke_color=kwargs.get('stroke_color', 'black'),
        stroke_width=kwargs.get('stroke_width', 2),
        font='Noto-Sans-CJK-JP',  # 한글 지원
        method='caption',
        size=(clip.w * 0.9, None)
    )
    
    # 위치 설정
    positions = {
        "bottom": ('center', 'bottom'),
        "top": ('center', 'top'),
        "center": ('center', 'center')
    }
    
    txt_clip = txt_clip.set_position(positions.get(position, position))
    txt_clip = txt_clip.set_duration(clip.duration)
    
    return CompositeVideoClip([clip, txt_clip])

print("✅ 편집 함수 로드 완료")

## 7️⃣ 메인 영상 생성

In [ ]:
def create_video_from_images(image_files, config):
    """이미지들로부터 영상 생성"""
    clips = []
    
    print(f"🎬 {len(image_files)}개 이미지로 영상 생성 시작...")
    
    for idx, img_path in enumerate(image_files):
        print(f"  처리 중: {idx+1}/{len(image_files)} - {os.path.basename(img_path)}")
        
        # 기본 클립 생성
        duration = config['duration_per_image']
        
        # Ken Burns 효과
        if config['enable_ken_burns']:
            clip = create_ken_burns_effect(img_path, duration, config['zoom_ratio'])
        else:
            clip = ImageClip(img_path, duration=duration)
        
        # 해상도 조정
        clip = clip.resize(config['resolution'])
        
        # 색상 필터
        if config['color_filter'] != 'none':
            clip = apply_color_filter(clip, config['color_filter'])
        
        # 자막 추가 (옵션)
        if config['add_captions'] and idx < len(CAPTIONS):
            clip = add_text_overlay(
                clip, 
                CAPTIONS[idx],
                position=config['caption_position'],
                font_size=config['font_size'],
                font_color=config['font_color'],
                stroke_color=config['stroke_color'],
                stroke_width=config['stroke_width']
            )
        
        clips.append(clip)
    
    # 클립 연결 (전환 효과 포함)
    if config['transition_type'] != 'none' and len(clips) > 1:
        final_clips = [clips[0]]
        for i in range(1, len(clips)):
            # 전환 효과로 연결
            trans_duration = config['transition_duration']
            clips[i] = clips[i].set_start(
                final_clips[-1].end - trans_duration
            ).crossfadein(trans_duration)
            final_clips.append(clips[i])
        
        final_video = CompositeVideoClip(final_clips)
    else:
        final_video = concatenate_videoclips(clips)
    
    # 색보정
    final_video = final_video.fx(
        vfx.colorx, 
        config.get('brightness', 1.0)
    )
    
    return final_video

# 자막 텍스트 (옵션 - 이미지별로 커스터마이즈)
CAPTIONS = [
    "첫 번째 장면",
    "두 번째 장면",
    "세 번째 장면",
    # ... 더 추가
]

# 영상 생성 실행
if image_files:
    video = create_video_from_images(image_files, VIDEO_CONFIG)
    print(f"✅ 영상 생성 완료: {video.duration}초")
else:
    print("⚠️ 이미지가 없습니다. Chrome Extension에서 먼저 업로드하세요.")

## 8️⃣ BGM 및 오디오 추가

In [ ]:
# 무료 BGM 다운로드 또는 생성
def add_background_music(video, bgm_source=None, volume=0.3, fade=True):
    """배경음악 추가"""
    
    if bgm_source is None:
        # 기본 무료 BGM 생성 (사인파 기반 앰비언트)
        print("🎵 기본 BGM 생성 중...")
        from moviepy.audio.AudioClip import AudioClip
        
        def make_ambient(t):
            """앰비언트 사운드 생성"""
            frequencies = [110, 220, 330]  # A2, A3, E4
            signal = np.zeros(len(t) if isinstance(t, np.ndarray) else 1)
            for freq in frequencies:
                signal += np.sin(2 * np.pi * freq * t) * 0.1
            return signal * volume
        
        audio = AudioClip(make_ambient, duration=video.duration)
        
    elif bgm_source.startswith('http'):
        # URL에서 다운로드
        print(f"🎵 BGM 다운로드: {bgm_source}")
        import requests
        response = requests.get(bgm_source)
        temp_audio = '/content/temp_bgm.mp3'
        with open(temp_audio, 'wb') as f:
            f.write(response.content)
        audio = AudioFileClip(temp_audio)
        
    else:
        # 로컬 파일
        audio = AudioFileClip(bgm_source)
    
    # 영상 길이에 맞춤
    if audio.duration > video.duration:
        audio = audio.subclip(0, video.duration)
    else:
        # 반복
        audio = audio.loop(duration=video.duration)
    
    # 볼륨 조절
    audio = audio.volumex(volume)
    
    # 페이드 인/아웃
    if fade:
        audio = audio.audio_fadein(2).audio_fadeout(2)
    
    # 영상에 오디오 추가
    final_video = video.set_audio(audio)
    return final_video

# BGM 추가 (옵션)
if VIDEO_CONFIG.get('bgm_url') or True:  # 기본 BGM 사용
    video = add_background_music(
        video, 
        VIDEO_CONFIG.get('bgm_url'),
        VIDEO_CONFIG.get('bgm_volume', 0.3),
        VIDEO_CONFIG.get('fade_audio', True)
    )
    print("✅ BGM 추가 완료")

## 9️⃣ 최종 렌더링 및 저장

In [ ]:
# 최종 영상 렌더링
output_filename = f"edited_video_{SESSION_ID}.{VIDEO_CONFIG['output_format']}"
output_path = f"{WORK_DIR}/output/{output_filename}"

# 품질 설정
quality = QUALITY_PRESETS[VIDEO_CONFIG['quality']]

print(f"🎬 최종 렌더링 시작... (품질: {VIDEO_CONFIG['quality']})")
print(f"   출력: {output_path}")

# 렌더링
video.write_videofile(
    output_path,
    fps=VIDEO_CONFIG['fps'],
    codec=VIDEO_CONFIG['codec'],
    bitrate=quality['bitrate'],
    audio_codec='aac',
    audio_bitrate='192k',
    preset='medium',  # ultrafast, fast, medium, slow
    threads=4,
    logger=None  # 로그 숨김
)

print(f"✅ 렌더링 완료!")
print(f"📁 파일 크기: {os.path.getsize(output_path) / (1024*1024):.1f} MB")

# 미리보기
display(Video(output_path, embed=True, width=720))

## 🔟 고급 편집 기능 (선택)

In [ ]:
# 추가 편집 기능들

def add_subtitles_from_srt(video, srt_file):
    """SRT 자막 파일 추가"""
    from moviepy.video.tools.subtitles import SubtitlesClip
    generator = lambda txt: TextClip(txt, font='Noto-Sans-CJK-JP', 
                                     fontsize=40, color='white')
    subtitles = SubtitlesClip(srt_file, generator)
    return CompositeVideoClip([video, subtitles])

def add_watermark(video, watermark_path, position=('right', 'bottom')):
    """워터마크 추가"""
    watermark = ImageClip(watermark_path).resize(height=50)
    watermark = watermark.set_duration(video.duration).set_position(position)
    return CompositeVideoClip([video, watermark])

def create_split_screen(clips, layout='2x2'):
    """분할 화면 생성"""
    if layout == '2x2' and len(clips) >= 4:
        return clips_array([[clips[0], clips[1]],
                           [clips[2], clips[3]]])
    elif layout == '1x2' and len(clips) >= 2:
        return clips_array([[clips[0], clips[1]]])
    return clips[0]

def add_speed_effect(clip, speed_factor=1.0):
    """속도 조절 (슬로모션/배속)"""
    return clip.fx(vfx.speedx, speed_factor)

def add_reverse_effect(clip):
    """역재생 효과"""
    return clip.fx(vfx.time_mirror)

def stabilize_video(video_path):
    """영상 안정화 (OpenCV)"""
    # OpenCV 기반 안정화 코드
    pass

print("✅ 고급 편집 함수 로드 완료")

## 1️⃣1️⃣ Chrome Extension으로 결과 전송

In [ ]:
# Extension으로 결과 전송을 위한 메타데이터 생성
result_meta = {
    "session_id": SESSION_ID,
    "status": "completed",
    "output_file": output_path,
    "download_url": f"https://drive.google.com/file/d/{output_path}",  # Drive 공유 링크
    "duration": video.duration,
    "resolution": VIDEO_CONFIG['resolution'],
    "file_size_mb": os.path.getsize(output_path) / (1024*1024),
    "created_at": datetime.now().isoformat()
}

# 메타데이터 저장
meta_file = f"{WORK_DIR}/result_{SESSION_ID}.json"
with open(meta_file, 'w') as f:
    json.dump(result_meta, f, indent=2)

print("📤 Extension 전송용 메타데이터:")
print(json.dumps(result_meta, indent=2))

# QR 코드 생성 (모바일 다운로드용)
try:
    import qrcode
    qr = qrcode.QRCode(version=1, box_size=5, border=2)
    qr.add_data(result_meta['download_url'])
    qr.make(fit=True)
    qr_img = qr.make_image(fill_color="black", back_color="white")
    qr_path = f"{WORK_DIR}/qr_{SESSION_ID}.png"
    qr_img.save(qr_path)
    print(f"📱 QR 코드 생성: {qr_path}")
    display(IPImage(qr_path, width=200))
except:
    print("QR 코드 생성 스킵 (qrcode 라이브러리 필요)")

## 1️⃣2️⃣ 다운로드 및 공유

In [ ]:
# 직접 다운로드
from google.colab import files
print("💾 브라우저로 다운로드 시작...")
files.download(output_path)

# YouTube 업로드 준비 (메타데이터)
youtube_meta = {
    "title": f"Chrome Extension Video {SESSION_ID}",
    "description": f"Created with Chrome Extension + Colab\nDuration: {video.duration}s\nResolution: {VIDEO_CONFIG['resolution']}",
    "tags": ["chrome-extension", "colab", "ai-video", "moviepy"],
    "category": "22",  # People & Blogs
    "privacy": "unlisted"  # private, unlisted, public
}

print("\n📺 YouTube 업로드 메타데이터:")
print(json.dumps(youtube_meta, indent=2))

---\n\n## 📚 사용 가이드\n\n### Chrome Extension에서:\n1. 이미지 수집 완료\n2. 'Colab으로 편집' 버튼 클릭\n3. Session ID 복사\n4. Google Drive에 자동 업로드\n\n### Colab에서:\n1. 이 노트북 실행\n2. Session ID 입력\n3. 편집 설정 조정\n4. 렌더링 실행\n5. 결과 다운로드\n\n### 편집 옵션:\n- **전환 효과**: crossfade, slide, zoom\n- **모션**: Ken Burns, 패닝, 줌\n- **필터**: 시네마틱, 빈티지, 색온도\n- **텍스트**: 자막, 워터마크, 타이틀\n- **오디오**: BGM, 효과음, 나레이션\n\n### 팁:\n- GPU 사용 시 렌더링 5-10배 빠름\n- 고화질은 'high' 또는 'ultra' 품질 선택\n- 긴 영상은 부분별로 나눠서 작업\n\n---\n\n*Chrome Extension + Colab 무료 영상 편집 파이프라인 v1.0*